<a href="https://colab.research.google.com/github/nissi31/fitness-pose-app/blob/main/exercise_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics opencv-python tqdm -q

import cv2
import numpy as np
import json
import math
from ultralytics import YOLO
from google.colab import files
from tqdm import tqdm

print("Please upload your video file (MP4, AVI, etc.):")
uploaded = files.upload()
video_path = next(iter(uploaded))
print(f"Uploaded '{video_path}'")

model = YOLO('yolov8n-pose.pt')

cap = cv2.VideoCapture(video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_keypoints = []

for i in tqdm(range(frame_count), desc="Extracting Keypoints"):
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
        keypoints = results[0].keypoints.xy[0].cpu().numpy().tolist()
    else:
        keypoints = []
    frame_keypoints.append(keypoints)

cap.release()
print("Keypoint extraction complete.")

KEYPOINT_DICT = {
    'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_ear': 3, 'right_ear': 4,
    'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12,
    'left_knee': 13, 'right_knee': 14, 'left_ankle': 15, 'right_ankle': 16
}

def calculate_angle(a, b, c):
    """Calculates the angle at point b between points a, b, and c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return angle if angle <= 180 else 360 - angle

def detect_squats(keypoints_seq):
    # Required keypoints for squats
    lk, rk = KEYPOINT_DICT['left_knee'], KEYPOINT_DICT['right_knee']
    lh, rh = KEYPOINT_DICT['left_hip'], KEYPOINT_DICT['right_hip']
    la, ra = KEYPOINT_DICT['left_ankle'], KEYPOINT_DICT['right_ankle']

    reps, state, rep_accuracies = 0, "up", []
    angle_history = []

    threshold_down, threshold_up = 160, 100

    for kpts in keypoints_seq:
        if not kpts: continue

        avg_knee_angle = (calculate_angle(kpts[lh], kpts[lk], kpts[la]) +
                          calculate_angle(kpts[rh], kpts[rk], kpts[ra])) / 2

        if state == "up" and avg_knee_angle < threshold_up:
            state = "down"
            angle_history = []
        elif state == "down" and avg_knee_angle > threshold_down:
            state = "up"
            reps += 1
            min_angle = min(angle_history) if angle_history else threshold_up
            accuracy = np.clip(((140 - min_angle) / (140 - 90)) * 100, 0, 100)
            rep_accuracies.append(accuracy)

        if state == "down":
            angle_history.append(avg_knee_angle)

    final_accuracy = int(np.mean(rep_accuracies)) if rep_accuracies else 0
    return reps, final_accuracy

def detect_pushups(keypoints_seq):
    le, re = KEYPOINT_DICT['left_elbow'], KEYPOINT_DICT['right_elbow']
    ls, rs = KEYPOINT_DICT['left_shoulder'], KEYPOINT_DICT['right_shoulder']
    lw, rw = KEYPOINT_DICT['left_wrist'], KEYPOINT_DICT['right_wrist']

    reps, state, rep_accuracies = 0, "up", []
    angle_history = []

    threshold_down, threshold_up = 100, 160

    for kpts in keypoints_seq:
        if not kpts: continue

        avg_elbow_angle = (calculate_angle(kpts[ls], kpts[le], kpts[lw]) +
                           calculate_angle(kpts[rs], kpts[re], kpts[rw])) / 2

        if state == "up" and avg_elbow_angle < threshold_down:
            state = "down"
            angle_history = []
        elif state == "down" and avg_elbow_angle > threshold_up:
            state = "up"
            reps += 1
            min_angle = min(angle_history) if angle_history else threshold_down
            accuracy = np.clip(((160 - min_angle) / (160 - 90)) * 100, 0, 100)
            rep_accuracies.append(accuracy)

        if state == "down":
            angle_history.append(avg_elbow_angle)

    final_accuracy = int(np.mean(rep_accuracies)) if rep_accuracies else 0
    return reps, final_accuracy

def detect_pullups(keypoints_seq):
    n, lw, rw = KEYPOINT_DICT['nose'], KEYPOINT_DICT['left_wrist'], KEYPOINT_DICT['right_wrist']

    reps, state, rep_accuracies = 0, "down", []
    height_history = []

    for kpts in keypoints_seq:
        if not kpts: continue

        nose_y = kpts[n][1]
        avg_wrist_y = (kpts[lw][1] + kpts[rw][1]) / 2

        if state == "down" and nose_y < avg_wrist_y:
            state = "up"
            height_history = []
        elif state == "up" and nose_y > avg_wrist_y:
            state = "down"
            reps += 1
            # Accuracy is based on how high the nose got (lower y-value is higher)
            min_y_pos = min(height_history) if height_history else avg_wrist_y
            accuracy = np.clip(((avg_wrist_y - min_y_pos) / (kpts[KEYPOINT_DICT['left_shoulder']][1] - avg_wrist_y + 1e-6)) * 100, 0, 100)
            rep_accuracies.append(accuracy)

        if state == "up":
            height_history.append(nose_y)

    final_accuracy = int(np.mean(rep_accuracies)) if rep_accuracies else 0
    return reps, final_accuracy

def detect_jumping_jacks(keypoints_seq):
    ls, rs = KEYPOINT_DICT['left_shoulder'], KEYPOINT_DICT['right_shoulder']
    lw, rw = KEYPOINT_DICT['left_wrist'], KEYPOINT_DICT['right_wrist']
    la, ra = KEYPOINT_DICT['left_ankle'], KEYPOINT_DICT['right_ankle']
    n = KEYPOINT_DICT['nose']

    reps, state = 0, "in"
    rep_accuracies = []

    peak_arm_height = float('inf')
    peak_leg_width = 0

    for kpts in keypoints_seq:
        if not kpts: continue

        avg_wrist_y = (kpts[lw][1] + kpts[rw][1]) / 2
        avg_shoulder_y = (kpts[ls][1] + kpts[rs][1]) / 2
        ankle_dist = abs(kpts[la][0] - kpts[ra][0])
        shoulder_width = abs(kpts[ls][0] - kpts[rs][0])
        nose_y = kpts[n][1]

        is_out_pose = avg_wrist_y < avg_shoulder_y and ankle_dist > (shoulder_width * 1.5)
        is_in_pose = avg_wrist_y > avg_shoulder_y and ankle_dist < (shoulder_width * 1.2)

        if state == "in" and is_out_pose:
            state = "out"
            peak_arm_height = avg_wrist_y
            peak_leg_width = ankle_dist

        elif state == "out":
            peak_arm_height = min(peak_arm_height, avg_wrist_y)
            peak_leg_width = max(peak_leg_width, ankle_dist)

            if is_in_pose:
                state = "in"
                reps += 1
                arm_range = avg_shoulder_y - nose_y
                achieved_arm_range = avg_shoulder_y - peak_arm_height
                arm_accuracy = np.clip((achieved_arm_range / (arm_range + 1e-6)) * 100, 0, 100)

                target_leg_width = shoulder_width * 2.2
                leg_accuracy = np.clip((peak_leg_width / target_leg_width) * 100, 0, 100)

                rep_accuracies.append((arm_accuracy + leg_accuracy) / 2)

    final_accuracy = int(np.mean(rep_accuracies)) if rep_accuracies else 0
    return reps, final_accuracy


def analyze_exercise(frames_kpts, exercise):
    """Dispatcher function to call the correct exercise detection logic."""
    if exercise == "squat":
        return detect_squats(frames_kpts)
    elif exercise == "pushup":
        return detect_pushups(frames_kpts)
    elif exercise == "pullup":
        print("Note: Pull-up detection can be less accurate due to camera angles and bar occlusion.")
        return detect_pullups(frames_kpts)
    elif exercise == "jumpingjack":
        return detect_jumping_jacks(frames_kpts)
    else:
        return 0, 0

EXERCISE_OPTIONS = ["squat", "pushup", "pullup", "jumpingjack"]

print("\n" + "="*40)
print("Which exercise do you want to analyze?")
print(f"Options: {', '.join(EXERCISE_OPTIONS)}")
print("="*40)

chosen_exercise = ""
while chosen_exercise not in EXERCISE_OPTIONS:
    chosen_exercise = input("Enter your choice: ").strip().lower()
    if chosen_exercise not in EXERCISE_OPTIONS:
        print("Invalid choice. Please select from the list.")

reps, accuracy = analyze_exercise(frame_keypoints, chosen_exercise)

print("\n--- ANALYSIS COMPLETE ---")
result_text = (
    f"Exercise: {chosen_exercise.title()}\n"
    f"Repetitions: {reps}\n"
    f"Accuracy: {accuracy}%"
)
print(result_text)
print("---------------------------------")

with open("exercise_analysis_result.txt", "w") as f:
    f.write(result_text)
files.download("exercise_analysis_result.txt")

Please upload your video file (MP4, AVI, etc.):


Saving 32939_2.mp4 to 32939_2 (1).mp4
Uploaded '32939_2 (1).mp4'


🤖 Extracting Keypoints: 100%|██████████| 96/96 [00:01<00:00, 70.90it/s]


✅ Keypoint extraction complete.

Which exercise do you want to analyze?
Options: squat, pushup, pullup, jumpingjack
Enter your choice: squat

--- 📈 ANALYSIS COMPLETE 📈 ---
💪 Exercise: Squat
🔄 Repetitions: 1
🎯 Accuracy: 100%
---------------------------------


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install ultralytics opencv-python -q

import cv2
import numpy as np
import math
import os
import zipfile
from ultralytics import YOLO
from google.colab import files
from tqdm import tqdm

# Dictionary to map keypoint names to their indices from the YOLOv8 model
KEYPOINT_DICT = {
    'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_ear': 3, 'right_ear': 4,
    'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12,
    'left_knee': 13, 'right_knee': 14, 'left_ankle': 15, 'right_ankle': 16
}

# Helper function to calculate the angle between three points
def calculate_angle(a, b, c):
    """Calculates the angle at point b between points a and c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return angle if angle <= 180 else 360 - angle

# --- RULE ENGINES: Translating Fitness-AQA errors into code ---

def check_backsquat_form(kpts, state_tracker):
    feedback = []
    if state_tracker['state'] == "down":
        lk, rk = kpts[KEYPOINT_DICT['left_knee']], kpts[KEYPOINT_DICT['right_knee']]
        la, ra = kpts[KEYPOINT_DICT['left_ankle']], kpts[KEYPOINT_DICT['right_ankle']]
        if abs(lk[0] - rk[0]) < abs(la[0] - ra[0]) * 0.9: feedback.append("KNEES IN! PUSH THEM OUT.")
    rs, rh, rk = kpts[KEYPOINT_DICT['right_shoulder']], kpts[KEYPOINT_DICT['right_hip']], kpts[KEYPOINT_DICT['right_knee']]
    if calculate_angle(rs, rh, rk) < 70 and state_tracker['state'] == "down": feedback.append("CHEST FALLING! KEEP IT UP.")
    return feedback

def check_barbellrow_form(kpts, state_tracker):
    feedback = []
    if state_tracker['state'] == "up" and state_tracker.get('initial_torso_angle'):
        rs, rh, ra = kpts[KEYPOINT_DICT['right_shoulder']], kpts[KEYPOINT_DICT['right_hip']], kpts[KEYPOINT_DICT['right_ankle']]
        current_angle = calculate_angle(rs, rh, ra)
        if current_angle > state_tracker['initial_torso_angle'] + 15: feedback.append("DON'T USE YOUR BACK! KEEP TORSO STILL.")
    return feedback

def check_overheadpress_form(kpts, state_tracker):
    feedback = []
    if state_tracker['state'] == "up":
        current_hip_y = kpts[KEYPOINT_DICT['right_hip']][1]
        if current_hip_y > state_tracker.get('start_hip_y', 0) + 10: feedback.append("DON'T USE YOUR LEGS! STRICT PRESS.")
    return feedback

def check_jumpingjack_form(kpts, state_tracker):
    feedback = []
    # A simple rule: check if arms are fully extended
    if state_tracker['state'] == "out":
        ls, le, lw = kpts[KEYPOINT_DICT['left_shoulder']], kpts[KEYPOINT_DICT['left_elbow']], kpts[KEYPOINT_DICT['left_wrist']]
        elbow_angle = calculate_angle(ls, le, lw)
        if elbow_angle < 150: # If arms are bent
            feedback.append("EXTEND ARMS FULLY!")
    return feedback


def update_backsquat_state(kpts, tracker):
    lh, rh, lk, rk, la, ra = (kpts[KEYPOINT_DICT[name]] for name in ['left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle'])
    angle = (calculate_angle(lh, lk, la) + calculate_angle(rh, rk, ra)) / 2
    if tracker['state'] == "up" and angle < 160:
        tracker['state'] = "down"; tracker['min_angle_rep'] = angle
    elif tracker['state'] == "down":
        tracker['min_angle_rep'] = min(tracker['min_angle_rep'], angle)
        if angle > 165:
            tracker['state'] = "up"; tracker['reps'] += 1
            if tracker['min_angle_rep'] > 100: tracker['last_rep_feedback'] = "GO DEEPER!"
    return tracker

def update_barbellrow_state(kpts, tracker):
    ls, rs, le, re, lw, rw = (kpts[KEYPOINT_DICT[name]] for name in ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'])
    angle = (calculate_angle(ls, le, lw) + calculate_angle(rs, re, rw)) / 2
    if tracker['state'] == "down" and angle < 100:
        tracker['state'] = "up"
        rs, rh, ra = kpts[KEYPOINT_DICT['right_shoulder']], kpts[KEYPOINT_DICT['right_hip']], kpts[KEYPOINT_DICT['right_ankle']]
        tracker['initial_torso_angle'] = calculate_angle(rs, rh, ra)
    elif tracker['state'] == "up" and angle > 160:
        tracker['state'] = "down"; tracker['reps'] += 1; tracker['initial_torso_angle'] = None
    return tracker

def update_overheadpress_state(kpts, tracker):
    ls, rs, le, re, lw, rw = (kpts[KEYPOINT_DICT[name]] for name in ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'])
    angle = (calculate_angle(ls, le, lw) + calculate_angle(rs, re, rw)) / 2
    if tracker['state'] == "down" and angle < 100:
        tracker['state'] = "up"; tracker['start_hip_y'] = kpts[KEYPOINT_DICT['right_hip']][1]
    elif tracker['state'] == "up" and angle > 160:
        tracker['state'] = "down"; tracker['reps'] += 1
    return tracker

def update_jumpingjack_state(kpts, tracker):
    lw, rw = kpts[KEYPOINT_DICT['left_wrist']], kpts[KEYPOINT_DICT['right_wrist']]
    ls, rs = kpts[KEYPOINT_DICT['left_shoulder']], kpts[KEYPOINT_DICT['right_shoulder']]
    avg_wrist_y = (lw[1] + rw[1]) / 2
    avg_shoulder_y = (ls[1] + rs[1]) / 2
    if tracker['state'] == "in" and avg_wrist_y < avg_shoulder_y: tracker['state'] = "out"
    elif tracker['state'] == "out" and avg_wrist_y > avg_shoulder_y:
        tracker['state'] = "in"; tracker['reps'] += 1
    return tracker


def run_ai_coach():
    EXERCISES = ["BackSquat", "BarbellRow", "OverheadPress", "JumpingJack"]

    print("🤖 AI Fitness Coach Initialized!")
    print("Please upload your exercise video.")
    uploaded = files.upload()
    input_video_path = next(iter(uploaded))

    print("\nWhich exercise are you performing?")
    for i, ex in enumerate(EXERCISES): print(f"{i+1}. {ex}")
    choice = 0
    while choice not in range(1, len(EXERCISES) + 1):
        try: choice = int(input("Enter choice number: "))
        except: pass
    chosen_exercise = EXERCISES[choice - 1]

    cap = cv2.VideoCapture(input_video_path)
    frame_w, frame_h, fps = (int(cap.get(p)) for p in [cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS])
    output_video_path = f"coached_{chosen_exercise}.mp4"
    out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_w, frame_h))

    model = YOLO('yolov8s-pose.pt')
    initial_state = 'in' if chosen_exercise == 'JumpingJack' else ('down' if chosen_exercise in ['BarbellRow', 'OverheadPress'] else 'up')
    state_tracker = {'state': initial_state, 'reps': 0}

    update_state_func = globals()[f"update_{chosen_exercise.lower()}_state"]
    check_form_func = globals()[f"check_{chosen_exercise.lower()}_form"]

    # --- Main Processing Loop ---
    with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc=f"Analyzing {chosen_exercise}") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            results = model(frame, verbose=False)
            annotated_frame = results[0].plot()

            if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
                kpts = results[0].keypoints.xy[0].cpu().numpy().tolist()
                state_tracker = update_state_func(kpts, state_tracker)
                form_feedback = check_form_func(kpts, state_tracker)
            else:
                form_feedback = ["No person detected"]

            if state_tracker.get('last_rep_feedback'): form_feedback.append(state_tracker['last_rep_feedback'])

            cv2.rectangle(annotated_frame, (0, 0), (250, 80), (20, 20, 20), -1)
            cv2.putText(annotated_frame, "REPS", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            cv2.putText(annotated_frame, str(state_tracker['reps']), (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)

            y_pos = frame_h - 40
            for msg in reversed(form_feedback):
                cv2.rectangle(annotated_frame, (0, y_pos - 30), (frame_w, y_pos + 10), (0, 0, 0), -1)
                cv2.putText(annotated_frame, msg, (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                y_pos -= 45

            out.write(annotated_frame)
            if state_tracker.get('last_rep_feedback'): state_tracker['last_rep_feedback'] = None
            pbar.update(1)

    cap.release()
    out.release()
    print(f"\n✅ Analysis complete! Your coached video is ready.")
    files.download(output_video_path)

run_ai_coach()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralyti

Saving Squat_ Shallow vs Deep.mp4 to Squat_ Shallow vs Deep.mp4

Which exercise are you performing?
1. BackSquat
2. BarbellRow
3. OverheadPress
4. JumpingJack
Enter choice number: BackSquat
Enter choice number: 1


100%|██████████| 22.4M/22.4M [00:00<00:00, 92.5MB/s]
Analyzing BackSquat: 100%|██████████| 2168/2168 [00:47<00:00, 45.79it/s]


✅ Analysis complete! Your coached video is ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>